In [ ]:
# --- Colab setup (auto-inserted; no-op outside Colab). tag: colab-bootstrap ---
import sys
if "google.colab" in sys.modules:
    import os, subprocess, pathlib
    _slug = "aniryou/full-stack-agentic-engineer"
    _root = pathlib.Path("/content") / "full-stack-agentic-engineer"
    if not _root.exists():
        subprocess.run(["git", "clone", "--depth", "1", f"https://github.com/{_slug}.git", str(_root)], check=True)
    os.chdir(_root / "07-application-agent-framework/long-running-durable/long-running-agentic/long-running-agents-gcp/notebooks")
    for _c in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if (_c / "pyproject.toml").exists() or (_c / "setup.py").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(_c)]); break
        if (_c / "requirements.txt").exists():
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(_c / "requirements.txt")]); break
        if _c == _root:
            break
    if str(pathlib.Path.cwd()) not in sys.path:
        sys.path.insert(0, str(pathlib.Path.cwd()))


# 03 · Waiting for humans, undoing work, waking up on time — worked

Three patterns that share one idea: **the run parks in the store and costs nothing until something wakes it.**

| Pattern | Who wakes it | GCP trigger |
|---|---|---|
| Human-in-the-loop gate | a person clicking approve/reject | HTTP endpoint (IAP), or Cloud Workflows callback |
| Saga | the next Cloud Task | Cloud Tasks |
| Scheduled agent | a clock | Cloud Scheduler → Pub/Sub → Cloud Run |

In [1]:
import sys, os, json, warnings
warnings.filterwarnings("ignore")
ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))          # repo root when run from notebooks/
sys.path[:0] = [os.path.join(ROOT, "src"), os.path.join(ROOT, "notebooks")]

def show_journal(run):
    print(f"run {run.run_id}  status={run.status.value}  version={run.version}  steps={run.usage.steps}  tokens={run.usage.tokens}  cost=${run.usage.cost_usd:.4f}")
    for s in run.journal:
        out = json.dumps(s.output, default=str)[:70] if s.output is not None else (s.error or "")
        print(f"  [{s.index}] {s.kind.value:<6} {s.status.value:<7} {s.name:<18} key={s.idempotency_key or '-':<20} {out}")

In [2]:
from lragents.core import *
from lragents.patterns import DurableAgentLoop, SagaRunner, SagaStep, ScheduledAgent, approve, expire_stale_approvals, approval_link

clock = FakeClock()
gateway = PaymentGateway()
tools = ToolRegistry([Tool("charge_card", "Charge the card (needs approval)",
                           lambda a, c: gateway.charge(float(a["amount"]), idempotency_key=c.idempotency_key),
                           requires_approval=True)])
inbox = []                                                     # stands in for Slack / email
dispatcher = InMemoryDispatcher(clock=clock)
loop = DurableAgentLoop(store=InMemoryRunStore(clock=clock), llm=ScriptedLLM([Decision.call("charge_card", amount=1200.0), Decision.final("Paid.")]),
                        tools=tools, dispatcher=dispatcher, idempotency=InMemoryIdempotencyStore(), clock=clock,
                        notify=lambda run: inbox.append(approval_link("https://ops.example.com", run)))
run = loop.start("Pay invoice 1200")
dispatcher.drain(loop.handle)
parked = loop.store.get(run.run_id)
print("status:", parked.status.value, "| pending tasks:", dispatcher.pending(), "| LLM calls left:", loop.llm.remaining)
print("proposed call:", parked.waiting_on["tool"], parked.waiting_on["args"])
print("notification :", inbox[-1])

status: WAITING_HUMAN | pending tasks: 0 | LLM calls left: 1
proposed call: charge_card {'amount': 1200.0}
notification : https://ops.example.com/runs/run_30f9c22ced9b/approve?token=yBZfglE8RiA0IA62Iv_VqA


Nothing is scheduled, nothing is running, the model is not consulted again. The **exact** proposed call is stored with a one-time token.

## Approve → execute exactly that

In [3]:
approve(loop, run.run_id, parked.waiting_on["token"], approved=True, approver="cfo@example.com", comment="ok")
dispatcher.drain(loop.handle)
final = loop.store.get(run.run_id)
show_journal(final)
print("charges:", [(c.amount) for c in gateway.charges], "| LLM calls left after approval:", 0 if loop.llm.remaining == 0 else loop.llm.remaining)
approve(loop, run.run_id, parked.waiting_on["token"], True, "cfo@example.com")     # double click → no-op
print("charges after a second approve:", len(gateway.charges))

run run_30f9c22ced9b  status=SUCCEEDED  version=13  steps=3  tokens=720  cost=$0.0000
  [0] llm    done    decide             key=-                    {"kind": "tool_call", "tool": "charge_card", "args": {"amount": 1200.0
  [1] human  done    approval           key=-                    {"approved": true, "approver": "cfo@example.com", "comment": "ok"}
  [2] tool   done    charge_card        key=run_30f9c22ced9b:2   {"result": {"charge_id": "run_30f9c22ced9b:2", "amount": 1200.0, "repl
  [3] llm    done    decide             key=-                    {"kind": "final", "tool": null, "args": {}, "text": "Paid."}
charges: [1200.0] | LLM calls left after approval: 0
charges after a second approve: 1


The journal reads LLM → HUMAN → TOOL → LLM: the approved call was journaled as a STARTED intent and executed by the loop's normal recovery path. No re-planning between approval and execution.

## Reject → the model hears why

In [4]:
loop2 = DurableAgentLoop(store=InMemoryRunStore(clock=clock), llm=ScriptedLLM([Decision.call("charge_card", amount=1200.0), Decision.final("Understood, not paying.")]),
                         tools=tools, dispatcher=dispatcher, idempotency=InMemoryIdempotencyStore(), clock=clock)
run2 = loop2.start("Pay invoice 1200"); dispatcher.drain(loop2.handle)
approve(loop2, run2.run_id, loop2.store.get(run2.run_id).waiting_on["token"], approved=False, approver="cfo", comment="vendor not onboarded")
dispatcher.drain(loop2.handle)
print(loop2.store.get(run2.run_id).result, "| last message to the model:", loop2.llm.calls[-1][-1]["content"])

Understood, not paying. | last message to the model: HUMAN[approval]: {"approved": false, "approver": "cfo", "comment": "vendor not onboarded"}


## Every gate has a deadline
Cloud Scheduler hits `/internal/scheduler/tick`; stale approvals become an explicit outcome.

In [5]:
loop3 = DurableAgentLoop(store=InMemoryRunStore(clock=clock), llm=ScriptedLLM([Decision.call("charge_card", amount=5.0)]),
                         tools=tools, dispatcher=dispatcher, idempotency=InMemoryIdempotencyStore(), clock=clock)
run3 = loop3.start("Pay 5"); dispatcher.drain(loop3.handle)
print("expired now:", [r.run_id for r in expire_stale_approvals(loop3, ttl_s=72*3600)])
clock.advance(73*3600)
print("expired after 73h:", [(r.run_id, r.error) for r in expire_stale_approvals(loop3, ttl_s=72*3600, escalate=lambda r: print('  escalating', r.run_id))])

expired now: []
  escalating run_2601e0faf6bf
expired after 73h: [('run_2601e0faf6bf', 'approval timed out')]


The Cloud Workflows flavour of the same gate (create a callback endpoint, wait up to N days, resume) is in `infra/workflows/hitl_approval.yaml`:

In [6]:
print(open(os.path.join(ROOT, "infra", "workflows", "hitl_approval.yaml")).read())

# Human-in-the-loop gate as a Cloud Workflows execution.
#
# Why Workflows here: the execution itself is the durable wait. It can sit on
# a callback for days (execution limit is one year; await_callback defaults to
# 12h, so set the timeout explicitly), costs nothing while waiting, and
# retries/timeouts are declarative. The agent service never holds a thread.
#
#   1. create a callback endpoint unique to this execution
#   2. tell the agent service to notify the approver with that URL
#   3. wait up to 7 days for a POST to the callback
#   4. resume the run with the decision — or expire it explicitly
#
# Deploy:  gcloud workflows deploy hitl-approval --source=hitl_approval.yaml --service-account=$WF_SA
# Run:     gcloud workflows run hitl-approval --data='{"run_id":"run_x","proposal":{...},"approver":"cfo@example.com","service_url":"https://..."}'
main:
  params: [input]
  steps:
    - init:
        assign:
          - run_id: ${input.run_id}
          - service_url: ${input.service_

## Saga — undoing what already happened
Book flight → hotel → charge card. The card fails. Compensations run **in reverse**, one per wake-up, each journaled with an idempotency key. We also kill the process mid-compensation.

In [7]:
log = []
def act(name, fail=False):
    def _a(ctx, key):
        if fail: raise ToolError(f"{name}: upstream error")
        log.append(f"+{name}"); return {"ref": f"{name}-{key[-4:]}"}
    return _a
def comp(name):
    def _c(ctx, prior, key): log.append(f"-{name} (was {prior['ref']})")
    return _c
steps = [SagaStep("flight", act("flight"), comp("flight")), SagaStep("hotel", act("hotel"), comp("hotel")), SagaStep("card", act("card", fail=True), comp("card"))]
faults = FaultInjector(); d = InMemoryDispatcher()
saga = SagaRunner(store=InMemoryRunStore(), dispatcher=d, idempotency=InMemoryIdempotencyStore(), steps=steps, faults=faults)
run = saga.start({"trip": "SIN→AMS"})
for _ in range(3): d.deliver_one(saga.handle)
print("after 3 wake-ups:", saga.store.get(run.run_id).state["saga"]["phase"], log)
faults.crash_once_at("after_side_effect")          # die right after compensating the hotel
d.drain(saga.handle)
final = saga.store.get(run.run_id)
print(final.status.value, "|", final.error)
print("log:", log)
show_journal(final)

after 3 wake-ups: compensating ['+flight', '+hotel']
FAILED | card failed: card: upstream error
log: ['+flight', '+hotel', '-hotel (was hotel-tion)', '-flight (was flight-tion)']
run saga_a025270b2be0  status=FAILED  version=26  steps=0  tokens=0  cost=$0.0000
  [0] tool   done    flight:action      key=saga_a025270b2be0:flight:action {"result": {"ref": "flight-tion"}}
  [1] tool   done    hotel:action       key=saga_a025270b2be0:hotel:action {"result": {"ref": "hotel-tion"}}
  [2] tool   failed  card:action        key=saga_a025270b2be0:card:action card: upstream error
  [3] tool   done    hotel:compensate   key=saga_a025270b2be0:hotel:compensate {"compensated": "hotel"}
  [4] tool   done    flight:compensate  key=saga_a025270b2be0:flight:compensate {"compensated": "flight"}


`-hotel` appears once even though the process died right after it: the retry found the STARTED intent and the memoised compensation.

## Scheduled agent — leases and due times

In [8]:
clock = FakeClock(); store = InMemoryRunStore(clock=clock); seen = []
a = ScheduledAgent(store=store, work=lambda st: seen.append(("A", clock())) or "checked presale", interval_s=300, clock=clock, lease_ttl_s=120)
b = ScheduledAgent(store=store, work=lambda st: seen.append(("B", clock())) or "checked presale", interval_s=300, clock=clock, lease_ttl_s=120, worker_id="worker-2")
a.ensure("presale-monitor")
print("tick 1:", a.tick("presale-monitor"))
print("tick 1 again (Scheduler retried):", a.tick("presale-monitor").reason)
clock.advance(301)
store.acquire_lease("presale-monitor", "zombie-worker", 120)        # a crashed instance still holds the lease
print("tick 2 while zombie holds lease:", b.tick("presale-monitor").reason)
clock.advance(121)
print("tick 2 after TTL:", b.tick("presale-monitor"))
print("work executed:", seen)

tick 1: TickResult(ran=True, reason='ran', output='checked presale')
tick 1 again (Scheduler retried): not due until 1700000300
tick 2 while zombie holds lease: overlap: presale-monitor leased by zombie-worker until 1700000421
tick 2 after TTL: TickResult(ran=True, reason='ran', output='checked presale')
work executed: [('A', 1700000000.0), ('B', 1700000422.0)]


## Takeaways
* A parked run has **no process**; approvals, callbacks and clocks all resume it through the same store.
* *Approve what you execute, execute what was approved* — journal the approved call, don't re-plan.
* Sagas: compensations are first-class side effects with their own keys; a failed compensation is an **escalation**, never silence.
* Cron is not a coordination primitive; **lease + due-time** is.